## FraudShield AI — Day 5
### FastAPI Real-Time Fraud Scoring Endpoint

Author: Suman Das | PNB 11 yrs | MTech IAR Jadavpur University
Goal: Deploy XGBoost fraud detector as production REST API
      with real-time scoring, SHAP explanation and health check

## Cell 1 — Install + Auto Restart

In [ ]:
# ── Permanent Environment Setup ──────────────────────────
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install",
                "fastapi", "uvicorn", "pyngrok",
                "shap", "xgboost", "joblib",
                "scikit-learn", "nest-asyncio",
                "--quiet",
                "--disable-pip-version-check"],
               capture_output=True)

import os
os.kill(os.getpid(), 9)

## Cell 2 — Verify Libraries

In [16]:
# ── Verify All Libraries ──────────────────────────────────
import fastapi
import uvicorn
import shap
import xgboost
import joblib
import sklearn
import numpy as np
import pandas as pd
import nest_asyncio
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

nest_asyncio.apply()

print("=" * 45)
print("FRAUDSHIELD AI — DAY 5")
print("=" * 45)
print(f"FastAPI   : {fastapi.__version__} ✓")
print(f"XGBoost   : {xgboost.__version__} ✓")
print(f"SHAP      : {shap.__version__} ✓")
print(f"numpy     : {np.__version__} ✓")
print("=" * 45)
print("Ready to build FastAPI! 🚀")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FRAUDSHIELD AI — DAY 5
FastAPI   : 0.135.1 ✓
XGBoost   : 3.2.0 ✓
SHAP      : 0.51.0 ✓
numpy     : 2.0.2 ✓
Ready to build FastAPI! 🚀


## Cell 3 — Load Models + Prepare SHAP Explainer

In [17]:
# ── Load Models and Prepare Explainer ────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import os
import pandas as pd
import numpy as np
import joblib

# Define paths for models and data
model_path_base = "/content/drive/MyDrive/Fraud_Detection_Project/"
data_file_path  = "/content/drive/MyDrive/creditcard.csv"

# Verify data file existence
if not os.path.exists(data_file_path):
    raise FileNotFoundError(f"The file {data_file_path} was not found. Please ensure it is in your Google Drive at the specified path.")

# Verify model files existence
if not os.path.exists(model_path_base + "fraudshield_xgb_model.pkl"):
    raise FileNotFoundError(f"The file {model_path_base + 'fraudshield_xgb_model.pkl'} was not found. Please ensure it is in your Google Drive at the specified path.")
if not os.path.exists(model_path_base + "fraudshield_scaler.pkl"):
    raise FileNotFoundError(f"The file {model_path_base + 'fraudshield_scaler.pkl'} was not found. Please ensure it is in your Google Drive at the specified path.")

# Load model and scaler
xgb_model = joblib.load(model_path_base +
                         "fraudshield_xgb_model.pkl")
scaler    = joblib.load(model_path_base +
                         "fraudshield_scaler.pkl")

# Rebuild feature list
df = pd.read_csv(data_file_path)
df = df.drop_duplicates()
df['Log_Amount'] = np.log1p(df['Amount'])
df['Hour']       = (df['Time'] / 3600).astype(int) % 24

feature_cols = [c for c in df.columns
                if c not in ['Time', 'Amount', 'Class']]

# Build SHAP explainer
explainer = shap.TreeExplainer(xgb_model)

print("=" * 45)
print("MODELS LOADED")
print("=" * 45)
print(f"XGBoost model  : ✓")
print(f"Scaler         : ✓")
print(f"SHAP explainer : ✓")
print(f"Features       : {len(feature_cols)}")
print(f"Feature list   : {feature_cols}")
print("\n✓ Ready to build API!")

MODELS LOADED
XGBoost model  : ✓
Scaler         : ✓
SHAP explainer : ✓
Features       : 30
Feature list   : ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Log_Amount', 'Hour']

✓ Ready to build API!


## Cell 4 — FastAPI Application

In [18]:
# ── FastAPI Application ───────────────────────────────────
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Dict, Any
import uvicorn

app = FastAPI(
    title       = "FraudShield AI",
    description = "Real-time fraud detection API with "
                  "SHAP explainability",
    version     = "1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins     = ["*"],
    allow_credentials = True,
    allow_methods     = ["*"],
    allow_headers     = ["*"],
)

# ── Request/Response Models ───────────────────────────────
class TransactionRequest(BaseModel):
    V1: float;  V2: float;  V3: float;  V4: float
    V5: float;  V6: float;  V7: float;  V8: float
    V9: float;  V10: float; V11: float; V12: float
    V13: float; V14: float; V15: float; V16: float
    V17: float; V18: float; V19: float; V20: float
    V21: float; V22: float; V23: float; V24: float
    V25: float; V26: float; V27: float; V28: float
    Amount: float
    Time: float

class PredictionResponse(BaseModel):
    fraud_probability : float
    decision          : str
    confidence        : str
    top_features      : List[Dict[str, Any]]
    threshold_used    : float

class BatchRequest(BaseModel):
    transactions: List[TransactionRequest]

# ── Helper Functions ──────────────────────────────────────
THRESHOLD = 0.5

def preprocess_transaction(transaction: dict) -> np.ndarray:
    """Preprocess single transaction for prediction."""
    log_amount = np.log1p(transaction['Amount'])
    hour       = int(transaction['Time'] / 3600) % 24

    features = []
    for col in feature_cols:
        if col == 'Log_Amount':
            features.append(log_amount)
        elif col == 'Hour':
            features.append(hour)
        else:
            features.append(transaction[col])

    return np.array(features).reshape(1, -1)

def get_top_shap_features(shap_vals: np.ndarray,
                           n: int = 5) -> list:
    """Return top N features by SHAP magnitude."""
    indices = np.argsort(np.abs(shap_vals))[::-1][:n]
    return [
        {
            "feature"     : feature_cols[i],
            "shap_value"  : round(float(shap_vals[i]), 4),
            "direction"   : "toward fraud"
                            if shap_vals[i] > 0
                            else "toward legitimate"
        }
        for i in indices
    ]

# ── API Endpoints ─────────────────────────────────────────
@app.get("/")
def root():
    return {
        "service"     : "FraudShield AI",
        "version"     : "1.0.0",
        "status"      : "operational",
        "endpoints"   : ["/predict", "/predict/batch",
                         "/health", "/docs"]
    }

@app.get("/health")
def health_check():
    return {
        "status"   : "healthy",
        "model"    : "XGBoost",
        "features" : len(feature_cols),
        "explainability" : "SHAP TreeExplainer"
    }

@app.post("/predict", response_model=PredictionResponse)
def predict(transaction: TransactionRequest):
    """
    Score a single transaction for fraud.
    Returns probability, decision and SHAP explanation.
    """
    tx_dict    = transaction.dict()
    features   = preprocess_transaction(tx_dict)
    scaled     = scaler.transform(features)

    prob       = float(xgb_model.predict_proba(scaled)[0][1])
    decision   = "FRAUD" if prob >= THRESHOLD else "LEGITIMATE"
    confidence = ("HIGH"   if prob > 0.8 or prob < 0.2 else
                  "MEDIUM" if prob > 0.6 or prob < 0.4 else
                  "LOW")

    shap_vals     = explainer.shap_values(
        pd.DataFrame(scaled, columns=feature_cols))[0]
    top_features  = get_top_shap_features(shap_vals)

    return PredictionResponse(
        fraud_probability = round(prob, 4),
        decision          = decision,
        confidence        = confidence,
        top_features      = top_features,
        threshold_used    = THRESHOLD
    )

@app.post("/predict/batch")
def predict_batch(batch: BatchRequest):
    """Score multiple transactions in one request."""
    results = []
    for tx in batch.transactions:
        tx_dict  = tx.dict()
        features = preprocess_transaction(tx_dict)
        scaled   = scaler.transform(features)
        prob     = float(
            xgb_model.predict_proba(scaled)[0][1])
        results.append({
            "fraud_probability" : round(prob, 4),
            "decision"          : "FRAUD"
                                  if prob >= THRESHOLD
                                  else "LEGITIMATE"
        })
    return {
        "total"        : len(results),
        "fraud_count"  : sum(1 for r in results
                             if r["decision"] == "FRAUD"),
        "results"      : results
    }

print("=" * 45)
print("FASTAPI APP CONFIGURED")
print("=" * 45)
print("Endpoints:")
print("  GET  /          — API info")
print("  GET  /health    — Health check")
print("  POST /predict   — Single transaction")
print("  POST /predict/batch — Batch scoring")
print("  GET  /docs      — Swagger UI")
print("\n✓ App ready to launch!")

FASTAPI APP CONFIGURED
Endpoints:
  GET  /          — API info
  GET  /health    — Health check
  POST /predict   — Single transaction
  POST /predict/batch — Batch scoring
  GET  /docs      — Swagger UI

✓ App ready to launch!


## Cell 5 — Launch API with ngrok Public UR

In [22]:
# ── Launch API with ngrok ─────────────────────────────────
from pyngrok import ngrok
import threading
import time

# ── Get ngrok auth token ──────────────────────────────────
# Free token from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "3BJETTMSzimiWFkr2nYVbveSGEZ_6DL3fJ75n9Btzp4QouJhH"  # ← paste your token

ngrok.set_auth_token(NGROK_TOKEN)

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000,
                log_level="error")

# Start server in background thread
thread = threading.Thread(target=run_server,
                          daemon=True)
thread.start()
time.sleep(3)

# Create public URL
public_url = ngrok.connect(8000)

print("=" * 55)
print("FRAUDSHIELD AI — API LIVE!")
print("=" * 55)
print(f"Public URL  : {public_url}")
print(f"Swagger UI  : {public_url}/docs")
print(f"Health      : {public_url}/health")
print(f"Predict     : {public_url}/predict")
print("=" * 55)
print("API is running — keep this cell active!")

ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


FRAUDSHIELD AI — API LIVE!
Public URL  : NgrokTunnel: "https://unexpecting-debatably-hee.ngrok-free.dev" -> "http://localhost:8000"
Swagger UI  : NgrokTunnel: "https://unexpecting-debatably-hee.ngrok-free.dev" -> "http://localhost:8000"/docs
Health      : NgrokTunnel: "https://unexpecting-debatably-hee.ngrok-free.dev" -> "http://localhost:8000"/health
Predict     : NgrokTunnel: "https://unexpecting-debatably-hee.ngrok-free.dev" -> "http://localhost:8000"/predict
API is running — keep this cell active!


## Cell 6 — Test API Endpoints

In [26]:
import requests
import json

BASE_URL = public_url.public_url # Correctly access the public URL string

# ── Test 1 ─ Health check ──────────────────
print("=" * 50)
print("TEST 1 ─ HEALTH CHECK")
print("=" * 50)
response = requests.get(f"{BASE_URL}/health")
print(json.dumps(response.json(), indent=2))

# ── Test 2 ─ Legitimate transaction ────────────────
print("\n" + "=" * 50)
print("TEST 2 ─ LEGITIMATE TRANSACTION")
print("=" * 50)

legit_tx = {
    "V1": -1.36, "V2": -0.07, "V3": 2.54,
    "V4": 1.38,  "V5": -0.34, "V6": 0.46,
    "V7": 0.24,  "V8": 0.10,  "V9": 0.36,
    "V10": 0.09, "V11": -0.55,"V12": -0.62,
    "V13": -0.99,"V14": -0.31,"V15": 1.47,
    "V16": -0.47,"V17": 0.21, "V18": 0.03,
    "V19": 0.40, "V20": 0.25, "V21": -0.02,
    "V22": 0.28, "V23": -0.11,"V24": 0.07,
    "V25": 0.13, "V26": -0.19,"V27": 0.13,
    "V28": -0.02,"Amount": 149.62, "Time": 0.0
}

response = requests.post(f"{BASE_URL}/predict",
                         json=legit_tx)
result   = response.json()
print(f"Decision          : {result['decision']}")
print(f"Fraud Probability : {result['fraud_probability']}")
print(f"Confidence        : {result['confidence']}")
print(f"\nTop SHAP Features:")
for f in result['top_features']:
    print(f"  {f['feature']:<15} : "
          f"{f['shap_value']:>8.4f} "
          f"({f['direction']})")

# ── Test 3 ─ Fraudulent transaction ──────────────
print("\n" + "=" * 50)
print("TEST 3 ─ FRAUDULENT TRANSACTION")
print("=" * 50)

# ── Get real fraud transaction from dataset ──────────
import pandas as pd
import numpy as np

df_test = pd.read_csv('/content/drive/MyDrive/creditcard.csv')
df_test = df_test.drop_duplicates()

# Get first real fraud transaction
real_fraud = df_test[df_test['Class']==1].iloc[0]

fraud_tx = {
    "V1" : float(real_fraud['V1']),
    "V2" : float(real_fraud['V2']),
    "V3" : float(real_fraud['V3']),
    "V4" : float(real_fraud['V4']),
    "V5" : float(real_fraud['V5']),
    "V6" : float(real_fraud['V6']),
    "V7" : float(real_fraud['V7']),
    "V8" : float(real_fraud['V8']),
    "V9" : float(real_fraud['V9']),
    "V10": float(real_fraud['V10']),
    "V11": float(real_fraud['V11']),
    "V12": float(real_fraud['V12']),
    "V13": float(real_fraud['V13']),
    "V14": float(real_fraud['V14']),
    "V15": float(real_fraud['V15']),
    "V16": float(real_fraud['V16']),
    "V17": float(real_fraud['V17']),
    "V18": float(real_fraud['V18']),
    "V19": float(real_fraud['V19']),
    "V20": float(real_fraud['V20']),
    "V21": float(real_fraud['V21']),
    "V22": float(real_fraud['V22']),
    "V23": float(real_fraud['V23']),
    "V24": float(real_fraud['V24']),
    "V25": float(real_fraud['V25']),
    "V26": float(real_fraud['V26']),
    "V27": float(real_fraud['V27']),
    "V28": float(real_fraud['V28']),
    "Amount": float(real_fraud['Amount']),
    "Time"  : float(real_fraud['Time'])
}

print(f"Real fraud V14 value: {real_fraud['V14']:.4f}")
print(f"Expected: V14 < -3 for high fraud probability")

# ── Now test with real fraud ──────────────────
response = requests.post(f"{BASE_URL}/predict",
                         json=fraud_tx)
result   = response.json()

print("\n" + "=" * 50)
print("TEST 3 ─ REAL FRAUD TRANSACTION FROM DATASET")
print("=" * 50)
print(f"Decision          : {result['decision']}")
print(f"Fraud Probability : {result['fraud_probability']}")
print(f"Confidence        : {result['confidence']}")
print(f"\nTop SHAP Features:")
for f in result['top_features']:
    print(f"  {f['feature']:<15} : "
          f"{f['shap_value']:>8.4f} "
          f"({f['direction']})")

# ── Batch test with real fraud ───────────────────
print("\n" + "=" * 50)
print("TEST 4 ─ BATCH SCORING (legit + real fraud)")
print("=" * 50)
batch    = {"transactions": [legit_tx, fraud_tx]}
response = requests.post(f"{BASE_URL}/predict/batch",
                         json=batch)
result   = response.json()
print(f"Total scored  : {result['total']}")
print(f"Fraud found   : {result['fraud_count']}")
for i, r in enumerate(result['results']):
    print(f"  Tx {i+1}: {r['decision']} "
          f"({r['fraud_probability']:.4f})")

# ── Test 4 ─ Batch scoring ───────────────────
print("\n" + "=" * 50)
print("TEST 4 ─ BATCH SCORING (2 transactions)")
print("=" * 50)
batch = {"transactions": [legit_tx, fraud_tx]}
response = requests.post(f"{BASE_URL}/predict/batch",
                         json=batch)
result   = response.json()
print(f"Total scored  : {result['total']}")
print(f"Fraud found   : {result['fraud_count']}")
for i, r in enumerate(result['results']):
    print(f"  Tx {i+1}: {r['decision']} "
          f"({r['fraud_probability']:.4f})")

TEST 1 ─ HEALTH CHECK
{
  "status": "healthy",
  "model": "XGBoost",
  "features": 30,
  "explainability": "SHAP TreeExplainer"
}

TEST 2 ─ LEGITIMATE TRANSACTION
Decision          : LEGITIMATE
Fraud Probability : 0.0
Confidence        : HIGH

Top SHAP Features:
  V11             :  -2.2182 (toward legitimate)
  V14             :  -2.1832 (toward legitimate)
  V3              :  -1.7059 (toward legitimate)
  V10             :  -1.3393 (toward legitimate)
  V20             :  -0.8474 (toward legitimate)

TEST 3 ─ FRAUDULENT TRANSACTION
Real fraud V14 value: -4.2893
Expected: V14 < -3 for high fraud probability

TEST 3 ─ REAL FRAUD TRANSACTION FROM DATASET
Decision          : FRAUD
Fraud Probability : 1.0
Confidence        : HIGH

Top SHAP Features:
  V14             :   4.2030 (toward fraud)
  V10             :   2.6184 (toward fraud)
  V4              :   1.3623 (toward fraud)
  Log_Amount      :  -1.0593 (toward legitimate)
  V12             :   1.0007 (toward fraud)

TEST 4 ─ BATCH S

## Day 5 Summary

API: FraudShield FastAPI — Real-Time Fraud Scoring

Endpoints Built:
- GET  /          — API info and version
- GET  /health    — Model health check
- POST /predict   — Single transaction scoring
- POST /predict/batch — Batch transaction scoring
- GET  /docs      — Auto-generated Swagger UI

Features:
- Real-time XGBoost fraud scoring
- SHAP explanation per prediction
- Confidence level (HIGH/MEDIUM/LOW)
- Top 5 feature contributions per decision
- Batch scoring for multiple transactions
- CORS enabled for web app integration

Production Skills Demonstrated:
- FastAPI REST API design
- Pydantic request/response validation
- Background thread deployment
- ngrok public URL tunneling
- Swagger auto-documentation

Next: Day 6 — Streamlit Fraud Analyst Dashboard